# ⭐ RigelSLM — Treino TXT/CSV/PDF no Google Colab (via Drive)

Este notebook monta seu Google Drive e executa o **treino CAUSAL (TXT)** do RigelSLM
com GPU (T4 ou superior), usando `treino.py` sobre pastas de texto em `dados/processed`.

**Versão:** 1.0.0 · **Data:** 04/08/2026
**Pipeline:** `treino.py` (causal — dados txt/csv/pdf convertidos para texto)
**Requer no Drive:** `rigelllm/` com os scripts + `tokenizer/tokenizer.json` + `dados/processed/` (pastas de texto)

## 1. Verificar Ambiente e GPU

In [ ]:
# 1. Verificar ambiente e GPU
import os, sys, subprocess, json, time
from datetime import datetime

try:
    import google.colab
    IS_COLAB = True
    print("✅ Ambiente: Google Colab")
except ImportError:
    IS_COLAB = False
    print("❌ Este notebook foi feito para o Google Colab.")
    sys.exit(1)

GPU_OK = False
GPU_NOME = ""
try:
    gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
    if gpu_info:
        GPU_NOME = gpu_info[0].strip()
        print(f"0001f3ae GPU detectada: {GPU_NOME}")
        GPU_OK = any(n in GPU_NOME for n in ["T4", "V100", "A100", "L4", "RTX", "P100"])
        if not GPU_OK:
            print("⚠️  GPU muito fraca. RigelSLM precisa de no mínimo T4.")
            print("   Vá em: Runtime → Alterar tipo de execução → T4 GPU")
    else:
        print("⚠️  Nenhuma GPU detectada.")
        print("   Vá em: Runtime → Alterar tipo de execução → T4 GPU")
except Exception:
    print("⚠️  Não foi possível detectar GPU.")

if not GPU_OK:
    print("\n" + "=" * 60)
    print("   ❌ TREINO CANCELADO: GPU inadequada ou ausente")
    print("   Ative T4 em: Runtime → Alterar tipo de execução")
    print("=" * 60)
    sys.exit(1)

print("✅ GPU OK. Pronto para treinar!")

## 2. Montar Google Drive

In [ ]:
# 2. Montar Google Drive e ir para a pasta do projeto
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/rigelllm"
os.chdir(DRIVE_PATH)

print(f"0001f4c1 Diretório de trabalho: {os.getcwd()}")
print(f"0001f4c4 Arquivos: {os.listdir()[:15]}...")
print()
print("0001f4c2 Pastas encontradas:")
for item in sorted(os.listdir()):
    if os.path.isdir(item):
        qtd = len(os.listdir(item)) if os.path.exists(item) else 0
        print(f"   0001f4c1 {item}/ ({qtd} arquivos)")
    else:
        print(f"   0001f4c4 {item}")

## 3. Sincronizar arquivos (direto do Drive)

Os arquivos do projeto (scripts, tokenizer, dados) estão no Drive em `rigelllm/`.
O treino será executado **diretamente do Drive** — sem copiar nada para o Colab.

> ⚠️ Os checkpoints/modelos são salvos em `rigelllm/modelo/` no seu Drive.
> Se o Colab desconectar, monte o Drive de novo e retome com `--resume`.

## 4. Instalar dependências

In [ ]:
# 4. Instalar dependências (só o necessário para o treino)
print("0001f4e6 Instalando dependências...")
!pip install --upgrade pip -q
!pip install torch tokenizers psutil tqdm -q
print("✅ Dependências instaladas")

import torch
print(f"0001f525 PyTorch {torch.__version__}")
print(f"0001f4a0 CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"0001f3ae GPU: {torch.cuda.get_device_name(0)}")

## 5. Configurar parâmetros do treino TXT

In [ ]:
# 5. Configurar parâmetros do treino TXT
# (gerado pelo dashboard) ↓

PASTA_DADOS = "dados/processed"          # pasta de texto (txt/csv/pdf convertidos)
MODO = "arquivo"                          # arquivo | dez | tudo | tempo
MAX_ARQUIVOS = 1                          # usado em modo arquivo/dez (marcar 1 ou 10)
HORAS = 1                                 # usado em modo tempo (1, 2 ou 3 horas)
EPOCHS = 5                                # número de épocas
RESUME = True                             # continuar do checkpoint?

print("0001f4cb Parâmetros configurados:")
print(f"   0001f4c1 Dados: {PASTA_DADOS}")
print(f"   0001f504 Modo: {MODO} | max-arquivos: {MAX_ARQUIVOS} | horas: {HORAS}")
print(f"   0001f500 Épocas: {EPOCHS} | resume: {RESUME}")

## 6. Executar treino TXT (treino.py — causal)

In [ ]:
# 6. Executar treino TXT
print("=" * 60)
print("   0001f9e0 INICIANDO TREINO TXT")
print("=" * 60)

if not os.path.exists("treino.py"):
    print("❌ treino.py não encontrado no Drive!")
    sys.exit(1)

cmd = ["python", "-u", "treino.py", "--dados", PASTA_DADOS, "--no-interactive"]
if RESUME:
    cmd.append("--resume")
if EPOCHS and EPOCHS > 0:
    cmd += ["--epochs", str(EPOCHS)]
if MODO in ("arquivo", "dez"):
    cmd += ["--max-arquivos", str(MAX_ARQUIVOS)]

comando_final = cmd
if MODO == "tempo":
    # Linux `timeout`: para o treino sozinho após N horas
    comando_final = ["timeout", f"{HORAS}h"] + cmd

print(f"0001f680 Comando: {' '.join(comando_final)}")
print()

processo = subprocess.Popen(
    comando_final,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for linha in processo.stdout:
    print(linha, end="")
processo.wait()
print()
if processo.returncode == 0:
    print("✅ Treino concluído com sucesso!")
else:
    print(f"❌ Treino encerrou com código {processo.returncode} (pode ser timeout planejado)")

## 7. Resultados já no Drive

In [ ]:
# 7. Resultados já estão no Drive
print("0001f4c1 Como trabalhamos direto do Drive, os checkpoints e logs já são salvos em:")
print(f"   {DRIVE_PATH}/modelo/")
print(f"   {DRIVE_PATH}/logs/")
print()
if os.path.exists("modelo/modelo_melhor.pt"):
    mb = os.path.getsize("modelo/modelo_melhor.pt") / 1e6
    print(f"   ✅ modelo/modelo_melhor.pt ({mb:.0f} MB)")
if os.path.exists("modelo/modelo.pt"):
    mb = os.path.getsize("modelo/modelo.pt") / 1e6
    print(f"   ✅ modelo/modelo.pt ({mb:.0f} MB)")
print()
print("0001f4a1 Depois do treino, baixe modelo/modelo_melhor.pt e teste no seu computador.")

---
**Fim do notebook TXT.** Modelo salvo em `modelo/` no Drive. Baixe e teste localmente.